# sheet_operate — Phase 4 GRPO（可驗證 reward 強化學習）
從 `sft_v2` 的 LoRA 接續訓練。reward = Spreadsheet Gym 的執行驗證分數（本地計算，零 API 成本）。

**前置需求**：Drive 上已有 `adapter_sft_v2`（先跑完 colab_sft_lora.ipynb 的 sft_v2 輪）。

**設計要點**
- 任務池：v2 全部 240 題 ＋ 隨機抽 120 題 v1（防止簡單任務漂移）
- reward = 格式分(0.2) + 格級匹配分(×1.0) + 全對加成(+1.0)；GRPO 組內標準化
- KL 約束（beta）擋住 policy 離 SFT 起點太遠
- 模型產生的程式碼經 AST 安全檢查後在子行程沙盒執行（Colab VM 本身可拋棄）
- checkpoint 存 Drive，session 斷線重跑全部 cell 自動續訓


In [ ]:
# ===== 設定 =====
CFG = dict(
    repo_url   = "https://github.com/timmytsaa/sheet_operate.git",
    drive_root = "/content/drive/MyDrive/公司/未命名資料夾/sheet_operate",
    run_name   = "grpo_v3",
    sft_adapter = "adapter_sft_v4",      # 起點：可給資料夾（自動取最新 checkpoint）或確切路徑
    base_model = "Qwen/Qwen3-4B-Instruct-2507",
    max_seq_len = 6144,
    lora_r = 64, lora_alpha = 64,
    # --- GRPO ---
    num_generations = 8,                 # 每題採樣數（組大小）
    max_prompt_len = 3584,
    max_completion_len = 1024,
    temperature = 0.9,
    lr = 5e-6, beta = 0.04,              # RL 學習率要低；beta = KL 約束強度
    max_steps = 400, save_steps = 40,
    per_device_bs = 8, grad_accum = 4,   # 每個 optimizer step = 4 題 × 8 生成
    v1_mix = 120,                        # 任務池摻入的 v1 題數（防簡單任務漂移）
    eval_limit = None,
    seed = 3407,
)

In [ ]:
%%capture
# ===== 安裝依賴（unsloth GRPO 需要 vllm 做快速 rollout） =====
!pip install unsloth vllm
!pip install openpyxl formulas   # formulas：公式任務的 reward 與評測都靠它求值

In [ ]:
# ===== 掛載 Drive、取得 repo、重生任務 =====
import glob, os, shutil, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

os.makedirs(CFG["drive_root"], exist_ok=True)
REPO = "/content/sheet_operate"
MARKER = os.path.join("scripts", "gen_tasks.py")

def find_zip():
    for name in ("repo.zip", "sheet_operate_repo.zip"):
        p = os.path.join(CFG["drive_root"], name)
        if os.path.exists(p):
            return p
    hits = sorted(glob.glob(os.path.join(CFG["drive_root"], "*.zip")))
    return hits[0] if hits else None

def has_marker(path):
    return os.path.exists(os.path.join(path, MARKER))

# repo 已存在就一律 pull。舊版寫成 `if not has_marker(REPO)`，clone 成功後
# 就再也不會更新——推上去的修正 Colab 這邊永遠拿不到，踩過兩次。
if CFG["repo_url"]:
    try:
        if not os.path.exists(REPO):
            subprocess.run(["git", "clone", CFG["repo_url"], REPO], check=True)
        else:
            r = subprocess.run(["git", "-C", REPO, "pull"], capture_output=True, text=True)
            print("git pull:", (r.stdout or r.stderr).strip().splitlines()[-1] if (r.stdout or r.stderr) else "")
    except Exception as e:
        print("[提示] git 取得失敗，改試 Drive 的 zip：", e)
    if not has_marker(REPO):
        shutil.rmtree(REPO, ignore_errors=True)

if not has_marker(REPO):
    zp = find_zip()
    if zp:
        import zipfile
        shutil.rmtree(REPO, ignore_errors=True)
        with zipfile.ZipFile(zp) as z:
            z.extractall(REPO)

assert has_marker(REPO), "找不到可用的 repo"
sys.path.insert(0, REPO)
os.chdir(REPO)

def run_script(args):
    r = subprocess.run([sys.executable] + args, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
        raise RuntimeError(f"指令失敗：{' '.join(args)}")

V1_FAMS = ("filter_rows,sort_rows,groupby_summary,compute_column,total_row,format_style,"
           "clean_data,join_lookup,split_concat,top_n,composite,context_rule,large_table")
V2_FAMS = "chain_v2,cross_sheet,format_v2,column_ops,semantic_map,calc_chain"

# RL 任務池（由 seed 重生，結構與本機 train 一致）＋ 三個評測集（v1 凍結）
V1_SINGLE = ("filter_rows,sort_rows,groupby_summary,compute_column,total_row,"
             "format_style,clean_data,join_lookup,split_concat,top_n")
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train",
            "--families", V1_SINGLE, "--n", "60", "--seed", "20260811"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train",
            "--families", "composite", "--n", "90", "--seed", "20260811"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train",
            "--families", "context_rule", "--n", "60", "--seed", "20260811"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train",
            "--families", "large_table", "--n", "30", "--seed", "20260811"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train_v2",
            "--families", V2_FAMS, "--n", "40", "--seed", "20260812"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train_v3",
            "--families", "offset_layout", "--n", "60", "--seed", "20260814"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train_v4",
            "--families", "formula_write", "--n", "60", "--seed", "20260815"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train_v5",
            "--families", "terse_intent", "--n", "70", "--seed", "20260818"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train_v6",
            "--families", "dup_header,misaligned_merge,two_tier_header,pair_group",
            "--n", "20", "--seed", "20260822"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/train_v7",
            "--families", "diff_dirty_key,diff_nullkey,diff_dupkey,diff_carry_cols,diff_multicol",
            "--n", "20", "--seed", "20260822"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval",
            "--families", V1_FAMS, "--n", "8", "--seed", "900001"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_ood",
            "--families", V1_FAMS, "--n", "5", "--seed", "900002", "--schema", "hr"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v2",
            "--families", V2_FAMS, "--n", "8", "--seed", "900003"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v3",
            "--families", "offset_layout", "--n", "12", "--seed", "900004"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v4",
            "--families", "formula_write", "--n", "12", "--seed", "900005"])
run_script([os.path.join(REPO, "scripts", "gen_tasks.py"), "--out", "data/tasks/eval_v5",
            "--families", "terse_intent", "--n", "14", "--seed", "900006"])
CKPT_DIR = os.path.join(CFG["drive_root"], "ckpt_" + CFG["run_name"])
print("repo:", REPO)
print("checkpoints:", CKPT_DIR)

In [ ]:
# ===== 載入模型（vLLM 快速 rollout）＋接上 sft_v2 LoRA =====
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = CFG["base_model"],
    max_seq_length = CFG["max_seq_len"],
    load_in_4bit   = False,
    fast_inference = True,               # vLLM rollout
    max_lora_rank  = CFG["lora_r"],
    gpu_memory_utilization = 0.7,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = CFG["lora_r"], lora_alpha = CFG["lora_alpha"], lora_dropout = 0,
    bias = "none",
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = CFG["seed"],
)

# 載入 sft_v2 adapter 權重作為 RL 起點
import os
import glob as _g
from safetensors.torch import load_file
from peft.utils import set_peft_model_state_dict


def _find_adapter():
    """起點權重：可為確切資料夾，或含 checkpoint-N 的資料夾（取步數最大者）；
    找不到時再掃 drive_root 的兄弟資料夾。"""
    roots = [CFG["drive_root"]] + sorted(
        _g.glob(os.path.join(os.path.dirname(CFG["drive_root"].rstrip("/")), "*")))
    for root in roots:
        base = os.path.join(root, CFG["sft_adapter"])
        direct = os.path.join(base, "adapter_model.safetensors")
        if os.path.exists(direct):
            return direct
        cks = _g.glob(os.path.join(base, "checkpoint-*", "adapter_model.safetensors"))
        if cks:
            def step(p):
                tail = os.path.basename(os.path.dirname(p)).rsplit("-", 1)[1]
                return int(tail) if tail.isdigit() else -1
            return max(cks, key=step)
    return None


adapter_path = _find_adapter()
assert adapter_path, f"找不到起點 adapter：{CFG['sft_adapter']}（請先完成 sft_v3 訓練）"
result = set_peft_model_state_dict(model, load_file(adapter_path))
print(f"已載入起點權重：{adapter_path}")
print(f"未匹配的鍵：{len(result.unexpected_keys)}")

In [ ]:
# ===== RL 任務池與 prompt 資料集 =====
import json as _json
import random
from pathlib import Path
from datasets import Dataset
from sheetops.encoder import encode_workbook
from sheetops.prompts import SYSTEM_PROMPT, build_user_prompt

random.seed(CFG["seed"])
hard_dirs = []
# RL 主戰場。v6/v7 一定要在池子裡——否則 GRPO 只在 v1~v5 上優化，
# 這一輪新增的家族拿不到任何 RL 練習，還可能被推得更低（SFT 後 v6 只有 37.5%）。
for d in ("train_v2", "train_v3", "train_v4", "train_v5", "train_v6", "train_v7"):
    hard_dirs += sorted(p.parent for p in Path(f"data/tasks/{d}").glob("*/task.json"))
v1_dirs = sorted(p.parent for p in Path("data/tasks/train").glob("*/task.json"))
pool = hard_dirs + random.sample(v1_dirs, min(CFG["v1_mix"], len(v1_dirs)))
random.shuffle(pool)
print(f"RL 任務池：v2~v7 {len(hard_dirs)} + v1 {len(pool) - len(hard_dirs)} = {len(pool)} 題")

records = []
for td in pool:
    spec = _json.loads((td / "task.json").read_text(encoding="utf-8"))
    user = build_user_prompt(spec["instruction"],
                             encode_workbook(td / "start.xlsx"),
                             spec.get("context", ""))
    prompt = tokenizer.apply_chat_template(
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": user}],
        tokenize=False, add_generation_prompt=True)
    records.append({"prompt": prompt, "task_dir": str(td)})

train_ds = Dataset.from_list(records)
print(f"prompt 資料集：{len(train_ds)} 筆")

In [ ]:
# ===== Reward：格式分 ＋ Gym 執行驗證分 ＋ 方法懲罰（沙盒平行執行） =====
from concurrent.futures import ThreadPoolExecutor
from sheetops.env import solve_once
from sheetops.executor import extract_code

import sys as _sys, os as _os
_sys.path.insert(0, _os.path.join(REPO, "scripts"))
from merge_teacher import LITERAL_INDEX, SWALLOW   # 與 SFT 資料過濾同一套判準

# 為什麼要方法懲罰：Gym 驗證的是「輸出」不是「方法」。一段硬編 row[7] 的程式碼
# 只要剛好猜對欄號就拿滿分——SFT 與 DPO 階段辛苦過濾掉的壞習慣，會在 RL 階段
# 被重新獎勵回來。真實檔案上那個習慣的下場是靜默輸出空表（AVTC 兩個 M/S 欄）。
METHOD_PENALTY = 0.5

def _method_penalty(code_):
    """字面量欄位索引或靜默吞例外 → 扣分。row[0] 當空列守衛例外（參考解法也這樣寫）。"""
    pen = 0.0
    if LITERAL_INDEX.search(code_):
        pen += METHOD_PENALTY
    if SWALLOW.search(code_):
        pen += METHOD_PENALTY
    return pen

def _gym_one(args):
    completion, task_dir = args
    code_ = extract_code(completion)
    if not code_:
        return 0.0
    try:
        rep = solve_once(task_dir, code_, timeout=30)
    except Exception:
        return 0.0
    # env.step() 執行失敗時保留「未變動的工作簿」去驗證（多回合環境的正確語意），
    # 但當 reward 用就變成「崩潰＝原表沒動＝拿到部分分數」。實測過一次：
    # 一段 unpack 錯誤的程式碼在 eval 拿到 0.993。跑不起來就是 0。
    if not rep.get("exec_ok", True):
        return 0.0
    score = float(rep["score"]) + (1.0 if rep["full_match"] else 0.0)
    return score - _method_penalty(code_)

def gym_reward(prompts, completions, task_dir, **kwargs):
    with ThreadPoolExecutor(max_workers=8) as ex:
        return list(ex.map(_gym_one, zip(completions, task_dir)))

def format_reward(prompts, completions, **kwargs):
    return [0.2 if extract_code(c) else 0.0 for c in completions]


In [ ]:
# ===== GRPO 訓練（自動續訓） =====
from trl import GRPOConfig, GRPOTrainer
from transformers.trainer_utils import get_last_checkpoint

os.makedirs(CKPT_DIR, exist_ok=True)
args = GRPOConfig(
    output_dir = CKPT_DIR,
    learning_rate = CFG["lr"],
    lr_scheduler_type = "constant_with_warmup",
    warmup_steps = 10,
    beta = CFG["beta"],
    per_device_train_batch_size = CFG["per_device_bs"],
    gradient_accumulation_steps = CFG["grad_accum"],
    num_generations = CFG["num_generations"],
    max_prompt_length = CFG["max_prompt_len"],
    max_completion_length = CFG["max_completion_len"],
    temperature = CFG["temperature"],
    max_steps = CFG["max_steps"],
    save_steps = CFG["save_steps"],
    save_total_limit = 3,
    logging_steps = 5,
    bf16 = True,
    report_to = "none",
    seed = CFG["seed"],
)
trainer = GRPOTrainer(
    model = model,
    processing_class = tokenizer,
    reward_funcs = [format_reward, gym_reward],
    args = args,
    train_dataset = train_ds,
)
last_ckpt = get_last_checkpoint(CKPT_DIR)
if last_ckpt:
    print("=" * 70)
    print(f"⚠️  在 {CKPT_DIR} 找到既有 checkpoint：{os.path.basename(last_ckpt)}")
    print("    將『續訓這一輪』，上面載入的起點權重會被 checkpoint 狀態覆蓋。")
    print("    若你要的是「從新起點開始的新一輪」，請把 CFG['run_name'] 換成沒用過的名字後重跑。")
    print("=" * 70)
else:
    print(f"全新一輪（{CFG['run_name']}），起點：{adapter_path}")
trainer.train(resume_from_checkpoint = last_ckpt)
# 觀察重點：reward 均值應持續上升；completion 長度不應爆走（爆走=在鑽格式漏洞）

In [ ]:
# ===== 存 GRPO adapter =====
ADAPTER_DIR = os.path.join(CFG["drive_root"], "adapter_" + CFG["run_name"])
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("adapter saved to:", ADAPTER_DIR)

In [ ]:
# ===== 訓後三軌評測 ＋ 與 sft_v2 對比 =====
# 自帶 import：評測常常在「跳過訓練、只跑評測」的情境下單獨執行，
# 那時第 5 格（RL 任務池）沒跑過，Path / _json 都不存在。
from pathlib import Path
import json as _json
import os
FastLanguageModel.for_inference(model)

def run_gym_eval(tasks_dir, tag):
    task_dirs = sorted(p.parent for p in Path(tasks_dir).glob("*/task.json"))
    if CFG["eval_limit"]:
        task_dirs = task_dirs[:CFG["eval_limit"]]
    rows = []
    for i, td in enumerate(task_dirs):
        spec = _json.loads((td / "task.json").read_text(encoding="utf-8"))
        user = build_user_prompt(spec["instruction"],
                                 encode_workbook(td / "start.xlsx"),
                                 spec.get("context", ""))
        prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user", "content": user}],
            tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        out = model.generate(**inputs, max_new_tokens=1024, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        reply = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                                 skip_special_tokens=True)
        code_ = extract_code(reply)
        rep = solve_once(td, code_) if code_ else {"full_match": False, "score": 0.0}
        rows.append({"id": spec["id"], "family": spec["family"],
                     "pass": bool(rep["full_match"]), "score": rep["score"]})
        if (i + 1) % 12 == 0:
            print(f"  [{tag}] {i+1}/{len(task_dirs)}  pass: {sum(r['pass'] for r in rows)}")
    import pandas as pd
    df = pd.DataFrame(rows)
    print(f"[{tag}] pass@1 = {df['pass'].mean():.1%}")
    print(df.groupby("family")["pass"].mean().sort_values().to_string())
    return df

results = {}
for tasks_dir, tag in [("data/tasks/eval", "v1"), ("data/tasks/eval_ood", "ood"),
                       ("data/tasks/eval_v2", "v2"), ("data/tasks/eval_v3", "v3"),
                       ("data/tasks/eval_v4", "v4"),
                       ("data/tasks/eval_v5", "v5"),
                       ("data/tasks/eval_v6", "v6"), ("data/tasks/eval_v7", "v7")]:
    if not os.path.isdir(tasks_dir):
        continue
    df = run_gym_eval(tasks_dir, f"{CFG['run_name']}-{tag}")
    df.to_csv(os.path.join(CFG["drive_root"], f"eval_{CFG['run_name']}_{tag}.csv"), index=False)
    results[tag] = df["pass"].mean()

print("\n===== GRPO 成績 =====")
PREV = {"v1": 0.913, "ood": 0.954, "v2": 0.583, "v3": 1.000,
        "v4": 0.583, "v5": 1.000}          # 上一輪 sft_v4
for tag, v in results.items():
    base = PREV.get(tag)
    delta = f"（sft_v4 {base:.1%} → {v - base:+.1%}）" if base else ""
    print(f"  {tag}: {v:.1%} {delta}")
print("\n判讀：v2/v4 是本輪主目標（RL 直接對執行結果優化，應明顯上升）；"
      "v1/ood/v3 不得比 sft_v3 低 5pt 以上（漂移警戒線）。")

## 之後
- v2 大漲、v1 持平 → Phase 5 部署（merge → GGUF → 本地 Ollama）
- v2 沒動 → 檢查 reward 曲線與 rollout 樣本（temperature、group size 調參）
- v1 掉了 → 提高任務池 v1 比例或加大 beta
